In [ ]:
import os
import torch
import json
import numpy as np
import matplotlib.pyplot as plt
from itertools import islice
from cryo_sbi import CryoEmSimulator
from cryo_sbi.inference.models import build_models
import cryo_sbi.inference.train_nle_model as train_nle_model
import cryo_sbi.utils.estimator_utils as est_utils
from cryo_sbi.inference.priors import get_image_priors, PriorLoader
import cryo_sbi.utils.image_utils as img_utils
from cryo_sbi.utils.visualize_models import plot_model

### Make the models 

Here we are creating a simple molecular model. 
In our model we will have four pseudo atoms arranged in a rectangle. The model differs between the sidelength we are using. The atoms are placed at the corners of the rectangle. The distance between the atoms is the same for all atoms.
The goal is then to simulate cryo-EM images with these models and infer the distance between the two atoms from the images.

The first step is to create the models. We start by crating an array with the side length `side_length` between the atoms. We will use this array to create the models.
The models are created by placing pseudo atoms at the corners of the rectangle.

The models are saved into teh file `models.pt`.



In [ ]:
# load models from file
models = torch.load("hsp90_models.pt")
models.shape
print(models.shape)

In [ ]:
def analyze_model_connectivity(models, cutoff=5.0):
    """
    Analyze how many atom pairs are within cutoff distance.
    
    Args:
        models: [num_models, 3, N] tensor
        cutoff: distance threshold
    
    Returns:
        Statistics about connectivity
    """
    # Transpose to [num_models, N, 3]
    coords = models.transpose(1, 2)
    
    num_models, N, _ = coords.shape
    
    edges_per_model = []
    
    for i in range(num_models):
        coord = coords[i]  # [N, 3]
        
        # Compute pairwise distances
        coord_i = coord.unsqueeze(1)  # [N, 1, 3]
        coord_j = coord.unsqueeze(0)  # [1, N, 3]
        dist = torch.norm(coord_i - coord_j, dim=-1)  # [N, N]
        
        # Count edges (excluding self-loops)
        mask = (dist < cutoff) & (~torch.eye(N, dtype=torch.bool, device=dist.device))
        num_edges = mask.sum().item()
        
        edges_per_model.append(num_edges)
    
    edges_per_model = np.array(edges_per_model)
    
    print(f"Cutoff: {cutoff}")
    print(f"Number of atoms (N): {N}")
    print(f"Max possible edges per model (excluding self): {N * (N - 1)}")
    print(f"\nEdges within cutoff:")
    print(f"  Mean: {edges_per_model.mean():.1f}")
    print(f"  Std: {edges_per_model.std():.1f}")
    print(f"  Min: {edges_per_model.min()}")
    print(f"  Max: {edges_per_model.max()}")
    print(f"  Density: {edges_per_model.mean() / (N * (N - 1)) * 100:.2f}%")
    
    return edges_per_model

# Test various cutoffs
for cutoff in [5.0, 10.0, 15.0, 20.0]:
    analyze_model_connectivity(models, cutoff)
    print("\n" + "="*50 + "\n")

In [ ]:
def center_models(models):
    """
    Remove center of mass from each model.
    
    Args:
        models: torch.Tensor of shape [num_models, 3, N]
                where 3 = (x, y, z) and N = number of atoms
    
    Returns:
        centered_models: torch.Tensor of same shape, centered at origin
    """
    # Compute center of mass for each model
    # Mean over atoms (dim=2) -> [num_models, 3]
    com = models.mean(dim=2, keepdim=True)  # [num_models, 3, 1]
    
    # Subtract center of mass
    centered_models = models - com
    
    return centered_models

# center and save models
models = center_models(models)
torch.save(models, "models.pt")

In [ ]:
models.shape

In [ ]:
def visualize_conformations(models, indices=[0, 5, 10, 15, 19]):
    """Visualize selected conformations in 3D."""
    # Convert torch to numpy for plotting if needed
    if isinstance(models, torch.Tensor):
        models = models.numpy()
    
    fig = plt.figure(figsize=(20, 4))
    
    for plot_idx, conf_idx in enumerate(indices):
        ax = fig.add_subplot(1, 5, plot_idx + 1, projection='3d')
        
        coords = models[conf_idx].T  # [n_atoms, 3]
        ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], 
                  c=np.arange(len(coords)), cmap='viridis', s=20, alpha=0.6)
        
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        
        # Set equal aspect ratio
        max_range = np.array([coords[:, 0].max()-coords[:, 0].min(),
                             coords[:, 1].max()-coords[:, 1].min(),
                             coords[:, 2].max()-coords[:, 2].min()]).max() / 2.0
        mid_x = (coords[:, 0].max()+coords[:, 0].min()) * 0.5
        mid_y = (coords[:, 1].max()+coords[:, 1].min()) * 0.5
        mid_z = (coords[:, 2].max()+coords[:, 2].min()) * 0.5
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    plt.tight_layout()
    plt.show()
    
# Visualize (automatically converts to numpy for plotting)
visualize_conformations(models)

### Run first simulation

We will now simulate the cryo-EM images with our generated models.
The simulation is done by the class `CryoEmSimulator`. And the simulation is run by the function `simulate` function.
The class `CryoEmSimulator` takes as input a config file with the simulation parameters. The config file used here is `simulation_parameters.json`.

The following parameters are used in the simulation:

```
simulation_parameters.json

{
    "N_PIXELS": 128,
    "PIXEL_SIZE": 1.5,
    "SIGMA": [0.5, 5.0],
    "MODEL_FILE": "hsp90_models.pt",
    "SHIFT": 0.0,
    "DEFOCUS": [0.5, 2.0],
    "SNR": [0.01, 0.5],
    "AMP": 0.1,
    "B_FACTOR": [1.0, 1.0]
}
```

In [ ]:
simulator = CryoEmSimulator(
    "simulation_parameters.json"
)  # creating simulator with simulation parameters

In [ ]:
images, parameters = simulator.simulate(
    num_sim=10000, return_parameters=True
)  # simulating images and save parameters

In [ ]:
#print(parameters)
#side_length = parameters[0]  # extracting side_length from parameters
#snr = parameters[-1]  # extracting snr from parameters
print(images.shape)

#### Visualize the simulated images

In [ ]:
# save images first
torch.save(images, "images.pt")
fig, axes = plt.subplots(10, 10, figsize=(10, 10), sharex=True, sharey=True)
for idx, ax in enumerate(axes.flatten()):
    ax.imshow(images[idx], vmin=-3, vmax=3, cmap="gray")
    #ax.set_title(
    #    f"Side: {side_lengths[side_length[idx].round().long()].item():.2f}", fontsize=10
    #)
    ax.axis("off")

### Train cryoSBI posterior

We will now train the cryoSBI posterior to infer the distance between the atoms from the simulated images.
The training is done with the function `npe_train_no_saving` which simulates images and simultaneously trains the posterior.
The function takes as input the config file `training_parameters.json` which contains the training and neural network parameters.
The function also takes as input the config file `simulation_parameters.json` which contains the simulation parameters used to simulate the images.


```
training_parameters_nle.json
```

```
{
    "EMBEDDING_X": "RESNET18",
    "EMBEDDING_THETA": "GNN",
    "OUT_DIM_X": 256,
    "OUT_DIM_THETA": 256,
    "NUM_TRANSFORM": 5,
    "NUM_HIDDEN_FLOW": 10,
    "HIDDEN_DIM_FLOW": 256,
    "MODEL": "NSF",
    "LEARNING_RATE": 0.0003,
    "CLIP_GRADIENT": 5.0,
    "BATCH_SIZE": 256
}
```


In [ ]:
train_nle_model.nle_train_no_saving(
    "simulation_parameters.json",
    "training_parameters_nle.json",
    150,
    "tutorial_estimator.pt",  # name of the estimator file
    "tutorial.loss",  # name of the loss file
    n_workers=4,  # number of workers for data loading
    device="cuda",  # device to use for training and simulation
    saving_frequency=100,  # frequency of saving the model
    simulation_batch_size=2048,  # batch size for simulation
)

In [ ]:
plt.plot(torch.load("tutorial.loss"))
plt.xlabel("Epoch")
plt.ylabel("Loss")